# Tests: `fasterai.export.onnx_exporter` (source `nbs/export/onnx_exporter.ipynb`)

In [ ]:
from fastcore.test import *
import warnings
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from fasterai.export.onnx_exporter import *
from fasterai.export.onnx_exporter import (_has_package, _pt2e_translation_table, _activation_pairs,
                                           _rewrite_activations_uint8, _check_uint8_activations,
                                           _graph_constants, _ort_outputs, _output_difference)
import fasterai.quantize.quantizer  # registers the pt2e q/dq operators used below

In [ ]:
from fastcore.test import *

# --- the translation table covers every q/dq operator the pt2e flow can emit ---
if _has_package("onnxscript"):
    _table = _pt2e_translation_table()
    _ops = torch.ops.quantized_decomposed
    for _overload in (_ops.quantize_per_tensor.default, _ops.dequantize_per_tensor.default,
                      _ops.quantize_per_channel.default, _ops.dequantize_per_channel.default):
        assert _overload in _table, f"missing translation for {_overload}"
    # per-tensor and per-channel have different signatures (`axis` only on the latter)
    import inspect
    test_eq(list(inspect.signature(_table[_ops.quantize_per_channel.default]).parameters)[3], 'axis')
    # the translations are plain functions: an `@onnxscript.script` wrapper would return a tuple
    assert all(inspect.isfunction(fn) for fn in _table.values())

# --- QDQStats is a plain, serializable record ---
test_eq(QDQStats(1, 2, 3, 4, 5).as_dict(),
        {'n_quantize': 1, 'n_dequantize': 2, 'n_per_channel': 3, 'n_nonzero_zero_point': 4,
         'n_unquantized_conv_add': 5, 'n_uint8': 0})  # n_uint8 comes last, and defaults
test_eq(QDQStats(1, 2, 3, 4, 5, 6).n_uint8, 6)

# --- what export_qdq promises about uint8 activations is what the rewrite does, and nothing more ---
assert 'zero-point 128' in export_qdq.__doc__ and 'same scales' in export_qdq.__doc__
for _word in ('faster', 'speed', 'latency', 'fuse', '×'):
    assert _word not in export_qdq.__doc__, f"the docstring claims '{_word}'"

In [ ]:
# --- qdq_stats reads every way a zero-point can be written into a graph ---
import tempfile

if _has_package("onnx"):
    import onnx
    from onnx import TensorProto, helper, numpy_helper

    def _tiny_qdq(path, zero_point=None, as_constant=False, per_channel=False, from_input=False,
                  as_uint8=False, weight=False):
        "Hand-built QuantizeLinear/DequantizeLinear pair with a controllable zero-point"
        dtype = np.uint8 if as_uint8 else np.int8
        scale = np.full(4, 0.05, np.float32) if per_channel else np.array(0.05, np.float32)
        initializers = [numpy_helper.from_array(scale, "scale")]
        inputs = [helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 4])]
        outputs = [helper.make_tensor_value_info("y", TensorProto.FLOAT, [1, 4])]
        nodes, q_inputs = [], ["x", "scale"]
        if from_input:  # zero-point only known at runtime: unreadable from the graph
            inputs.append(helper.make_tensor_value_info("zp", TensorProto.INT8, []))
            q_inputs.append("zp")
        elif zero_point is not None:
            zp = np.full(4, zero_point, dtype) if per_channel else np.array(zero_point, dtype)
            if as_constant:
                nodes.append(helper.make_node("Constant", [], ["zp"],
                                              value=numpy_helper.from_array(zp, "zp_value")))
            else:
                initializers.append(numpy_helper.from_array(zp, "zp"))
            q_inputs.append("zp")
        attrs = {"axis": 1} if per_channel else {}
        nodes += [helper.make_node("QuantizeLinear", q_inputs, ["q"], **attrs),
                  helper.make_node("DequantizeLinear", ["q"] + q_inputs[1:], ["y"], **attrs)]
        if weight:  # a weight constant, read through a DequantizeLinear of its own
            initializers += [numpy_helper.from_array(np.arange(4, dtype=np.int8), "w"),
                             numpy_helper.from_array(np.array(0.02, np.float32), "w_scale"),
                             numpy_helper.from_array(np.array(0, np.int8), "w_zp")]
            nodes.append(helper.make_node("DequantizeLinear", ["w", "w_scale", "w_zp"], ["w_out"]))
            outputs.append(helper.make_tensor_value_info("w_out", TensorProto.FLOAT, [4]))
        graph = helper.make_graph(nodes, "qdq", inputs, outputs, initializer=initializers)
        model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 18)])
        onnx.checker.check_model(model)
        onnx.save(model, str(path))
        return path

    with tempfile.TemporaryDirectory() as _tmp:
        _p = Path(_tmp)
        # zero-point stored as an initializer: the pin CAN fail
        test_eq(qdq_stats(_tiny_qdq(_p/'zero.onnx', zero_point=0)).as_dict(),
                {'n_quantize': 1, 'n_dequantize': 1, 'n_per_channel': 0, 'n_nonzero_zero_point': 0,
                 'n_unquantized_conv_add': 0, 'n_uint8': 0})
        test_eq(qdq_stats(_tiny_qdq(_p/'nonzero.onnx', zero_point=7)).n_nonzero_zero_point, 2)
        # zero-point produced by a Constant node
        test_eq(qdq_stats(_tiny_qdq(_p/'const.onnx', zero_point=3, as_constant=True)).n_nonzero_zero_point, 2)
        # zero-point omitted altogether: implicitly zero, and never counted as unsigned
        test_eq(qdq_stats(_tiny_qdq(_p/'omitted.onnx')).n_nonzero_zero_point, 0)
        test_eq(qdq_stats(_tiny_qdq(_p/'omitted.onnx')).n_uint8, 0)
        # an unsigned pair: 128 is a zero-point like any other, and its type is what n_uint8 reads
        test_eq(qdq_stats(_tiny_qdq(_p/'uint8.onnx', zero_point=128, as_uint8=True)).as_dict(),
                {'n_quantize': 1, 'n_dequantize': 1, 'n_per_channel': 0, 'n_nonzero_zero_point': 2,
                 'n_unquantized_conv_add': 0, 'n_uint8': 2})
        # per-channel scales are detected from the scale itself, not from the `axis` attribute
        test_eq(qdq_stats(_tiny_qdq(_p/'channel.onnx', zero_point=0, per_channel=True)).n_per_channel, 2)
        # a zero-point that is not a graph constant must raise, not be counted as zero
        with ExceptionExpected(ValueError, regex="zero_point"):
            qdq_stats(_tiny_qdq(_p/'runtime.onnx', from_input=True))

In [ ]:
# --- the uint8 rewrite moves the activation pairs, and only those ---
if _has_package("onnx"):
    with tempfile.TemporaryDirectory() as _tmp:
        _p, _x = Path(_tmp), torch.randn(1, 4)

        def _accepted(model, path, run=True):
            "Write a hand-built graph, and make sure ONNX — and ONNX Runtime — take it as it is"
            onnx.save(model, str(path))
            onnx.checker.check_model(str(path))
            if run and _has_package("onnxruntime"): _ort_outputs(path, _x, optimize=False)
            return model.graph

        _model = onnx.load(str(_tiny_qdq(_p/'pair.onnx', zero_point=0, weight=True)))
        _graph = _model.graph
        test_eq([len(_readers) for _, _readers in _activation_pairs(_graph)], [1])  # the weight DQ is not one
        test_eq(_rewrite_activations_uint8(_graph), 1)
        _check_uint8_activations(_graph, 1)  # the post-condition reads the rewritten graph back
        _constants = _graph_constants(_graph)
        _moved = [n for n in _graph.node if n.input[0] in ('x', 'q')]
        test_eq(len(_moved), 2)
        for _node in _moved:
            _zp = numpy_helper.to_array(_constants[_node.input[2]])
            test_eq(_zp.dtype, np.uint8)
            test_eq(_zp.reshape(-1).tolist(), [128])
            test_eq(_node.input[1], 'scale')  # the scales are not touched
        test_close(float(numpy_helper.to_array(_constants['scale'])), 0.05)
        assert 'zp' not in _constants, "the int8 zero-point nothing reads any more was kept"
        # the weight pair is left exactly as it was written
        _weight = next(n for n in _graph.node if n.input[0] == 'w')
        test_eq(numpy_helper.to_array(_constants[_weight.input[2]]).dtype, np.int8)
        test_eq(numpy_helper.to_array(_constants[_weight.input[2]]).reshape(-1).tolist(), [0])
        # ...and what comes out is a graph ONNX accepts and ONNX Runtime runs, to the same outputs
        _accepted(_model, _p/'pair_uint8.onnx')
        if _has_package("onnxruntime"):
            test_eq(_output_difference(_ort_outputs(_p/'pair.onnx', _x, optimize=False),
                                       _ort_outputs(_p/'pair_uint8.onnx', _x, optimize=False)), None)

        # a graph with no activation pair at all: the rewrite is a no-op that still checks out
        _weights_only = helper.make_model(
            helper.make_graph([helper.make_node("DequantizeLinear", ["w", "w_scale", "w_zp"], ["w_out"])],
                              "weights", [],
                              [helper.make_tensor_value_info("w_out", TensorProto.FLOAT, [4])],
                              initializer=[numpy_helper.from_array(np.arange(4, dtype=np.int8), "w"),
                                           numpy_helper.from_array(np.array(0.02, np.float32), "w_scale"),
                                           numpy_helper.from_array(np.array(0, np.int8), "w_zp")]),
            opset_imports=[helper.make_opsetid("", 18)])
        test_eq(_activation_pairs(_weights_only.graph), [])
        test_eq(_rewrite_activations_uint8(_weights_only.graph), 0)
        _check_uint8_activations(_weights_only.graph, 0)
        test_eq(numpy_helper.to_array(_graph_constants(_weights_only.graph)['w_zp']).dtype, np.int8)
        _accepted(_weights_only, _p/'weights_only.onnx', run=False)  # it has no input to feed

        # a zero-point constant the weight pair shares: the rewrite writes a new one rather than
        # moving that one, which would take the weight pair with it
        _shared = onnx.load(str(_tiny_qdq(_p/'shared.onnx', zero_point=0, weight=True)))
        _shared.graph.initializer.remove(next(i for i in _shared.graph.initializer if i.name == 'w_zp'))
        next(n for n in _shared.graph.node if n.input[0] == 'w').input[2] = 'zp'
        test_eq(_rewrite_activations_uint8(_accepted(_shared, _p/'shared.onnx')), 1)
        _check_uint8_activations(_shared.graph, 1)
        test_eq(numpy_helper.to_array(_graph_constants(_shared.graph)['zp']).dtype, np.int8)
        _accepted(_shared, _p/'shared_uint8.onnx')

        # a weight reaching its DequantizeLinear through a Cast is a weight all the same
        _cast = onnx.load(str(_tiny_qdq(_p/'cast.onnx', zero_point=0, weight=True)))
        _cast_weight = next(n for n in _cast.graph.node if n.input[0] == 'w')
        _cast.graph.node.insert(0, helper.make_node("Cast", ["w"], ["w_cast"], to=TensorProto.INT8))
        _cast_weight.input[0] = 'w_cast'
        test_eq(_rewrite_activations_uint8(_accepted(_cast, _p/'cast.onnx')), 1)
        _check_uint8_activations(_cast.graph, 1)
        test_eq(numpy_helper.to_array(_graph_constants(_cast.graph)['w_zp']).dtype, np.int8)
        _accepted(_cast, _p/'cast_uint8.onnx')
        # ...and the post-condition reads BOTH halves: the pair's own DequantizeLinear has to be
        # unsigned, and that weight one has to stay signed
        _cast_dq = next(n for n in _cast.graph.node if n.input[0] == 'q')
        _unsigned, _cast_dq.input[2] = _cast_dq.input[2], 'w_zp'
        with ExceptionExpected(ValueError, regex="not 128 everywhere"):
            _check_uint8_activations(_cast.graph, 1)
        _cast_dq.input[2] = _unsigned
        _cast_weight.input[2] = _unsigned
        with ExceptionExpected(ValueError, regex="not 0 everywhere"):
            _check_uint8_activations(_cast.graph, 1)

        # each graph below is one ONNX accepts and runs: what refuses it is the rule, not the fixture.
        # a node that reads the quantized value without dequantizing it
        _read = onnx.load(str(_tiny_qdq(_p/'read.onnx', zero_point=0)))
        _read.graph.node.append(helper.make_node("Identity", ["q"], ["q_copy"]))
        with ExceptionExpected(ValueError, regex="without dequantizing"):
            _activation_pairs(_accepted(_read, _p/'read_more.onnx'))
        # the quantized value is itself a graph output
        _returned = onnx.load(str(_tiny_qdq(_p/'returned.onnx', zero_point=0)))
        _returned.graph.output.append(helper.make_tensor_value_info("q", TensorProto.INT8, [1, 4]))
        with ExceptionExpected(ValueError, regex="graph output"):
            _activation_pairs(_accepted(_returned, _p/'returned_q.onnx'))
        # a DequantizeLinear reading neither a weight constant nor a QuantizeLinear output
        _loose = onnx.load(str(_tiny_qdq(_p/'loose.onnx', zero_point=0)))
        _loose.graph.input.append(helper.make_tensor_value_info("qin", TensorProto.INT8, [1, 4]))
        _loose.graph.node.append(helper.make_node("DequantizeLinear", ["qin", "scale"], ["z"]))
        _loose.graph.output.append(helper.make_tensor_value_info("z", TensorProto.FLOAT, [1, 4]))
        with ExceptionExpected(ValueError, regex="neither a graph constant"):
            _activation_pairs(_accepted(_loose, _p/'loose_dq.onnx', run=False))  # it takes a second input
        # an affine pair: moving a zero-point that is not 0 would change what the graph computes
        with ExceptionExpected(ValueError, regex="symmetric=False"):
            _rewrite_activations_uint8(onnx.load(str(_tiny_qdq(_p/'affine.onnx', zero_point=7))).graph)
        # a zero-point known only while the graph runs cannot be moved either
        with ExceptionExpected(ValueError, regex="not a graph constant"):
            _rewrite_activations_uint8(onnx.load(str(_tiny_qdq(_p/'live.onnx', from_input=True))).graph)
        # ...and neither can one the graph leaves out, which reads as 0 in its own tensor's type
        with ExceptionExpected(ValueError, regex="omits its zero-point"):
            _rewrite_activations_uint8(onnx.load(str(_tiny_qdq(_p/'none.onnx'))).graph)

        # the post-condition is read off the graph, and none of these can pass it
        _checked = onnx.load(str(_tiny_qdq(_p/'checked.onnx', zero_point=0, weight=True)))
        test_eq(_rewrite_activations_uint8(_checked.graph), 1)
        with ExceptionExpected(ValueError, regex="carries 1 QuantizeLinear"):
            _check_uint8_activations(_checked.graph, 2)
        next(n for n in _checked.graph.node if n.op_type == 'QuantizeLinear').input[0] = 'w'
        with ExceptionExpected(ValueError, regex="quantizes the graph constant"):
            _check_uint8_activations(_checked.graph, 1)
        with ExceptionExpected(ValueError, regex="not 128 everywhere"):
            _check_uint8_activations(onnx.load(str(_tiny_qdq(_p/'still_int8.onnx', zero_point=0))).graph, 1)

In [ ]:
# --- end to end: a pt2e-quantized model exported, inspected and verified ---
from fasterai.quantize.quantizer import Quantizer, _HAS_PT2E

if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    import onnx

    class _TinyConvNet(nn.Module):
        "Conv-BN-ReLU-Conv-Pool-Linear network, small enough to quantize in a test"
        def __init__(self, n_classes=10):
            super().__init__()
            self.features = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                                          nn.Conv2d(8, 16, 3, padding=1), nn.AdaptiveAvgPool2d(1))
            self.head = nn.Sequential(nn.Flatten(), nn.Linear(16, n_classes))
        def forward(self, x): return self.head(self.features(x))
        def spread_head(self, batch):
            "Center the classifier on the mean feature of `batch`, so its argmax follows the input"
            # An untrained head answers the same class to every input: its bias swamps the differences
            # between feature vectors. Reading the logits from the feature DEVIATIONS is what spreads a
            # batch over classes; the weights stay exactly as initialized, since rescaling them moves
            # every logit by the same factor and leaves the argmax where it was.
            with torch.no_grad():
                self.head[1].bias.copy_(-(self.head[1].weight @ self.features(batch).flatten(1).mean(0)))
            return self

    torch.manual_seed(0)
    _calib = [(torch.randn(4, 3, 16, 16), torch.randint(0, 10, (4,))) for _ in range(4)]
    _sample = torch.randn(4, 3, 16, 16)

    # `verify_qdq` compares ARGMAX, and an untrained net answers the same class to every input: its
    # agreement would read 1.0 against any graph at all. `spread_head` is what gives the agreements
    # below something to measure, and the spread is asserted before they are read.
    # Measured here (torch 2.9.1, onnxruntime CPU): over the 32 probes the reference spans 8 classes
    # (5 over the eight-probe slice) and agrees with its exported graph on 30 of 32 (0.938). 0.9 where
    # this check used to read 0.99 is deliberate — 0.938 is what a reference whose predictions vary
    # actually reads, because it sits near decision boundaries where ONNX Runtime's INT8 kernels round
    # differently than PyTorch's; on 32 probes 0.9 tolerates at most 3 disagreements. The negative
    # control — the same probe read against ANOTHER model's graph, where the agreement collapses —
    # lives with the pt2e fixtures in quantize/quantizer.ipynb and is not duplicated here.
    _MIN_CLASSES, _MIN_SLICE_CLASSES = 4, 2
    _MIN_AGREEMENT, _MIN_SLICE_AGREEMENT = 0.9, 0.75
    _qmodel = Quantizer(backend='pt2e').quantize(
        _TinyConvNet().eval().spread_head(torch.randn(64, 3, 16, 16)), _calib)

    with tempfile.TemporaryDirectory() as _tmp:
        _path = export_qdq(_qmodel, _sample, Path(_tmp)/'qdq.onnx')
        assert _path.exists()

        _stats = qdq_stats(_path)
        assert _stats.n_quantize > 0 and _stats.n_dequantize > 0, _stats
        assert _stats.n_per_channel > 0, "per-channel weights did not survive the export"
        test_eq(_stats.n_nonzero_zero_point, 0)  # portability: zero_point == 0 everywhere

        _static_batch = _sample.shape[0]  # the exported graph only accepts the batch it was traced on
        _spread_probe = torch.randn(32, 3, 16, 16)
        _n_batches = _spread_probe.shape[0] // _static_batch
        with torch.no_grad():
            _preds = torch.cat([_qmodel(_c).argmax(-1)
                                for _c in _spread_probe.split(_static_batch)]).tolist()
        assert len(set(_preds)) >= _MIN_CLASSES, ("the reference predictions must span classes, or "
                                                  f"the agreement below proves nothing: {_preds}")
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            _agreement = verify_qdq(_qmodel, _path, _spread_probe, n_batches=_n_batches)
        assert _agreement >= _MIN_AGREEMENT, f"ONNX and PyTorch disagree too often: {_agreement}"
        assert not any('vacuous' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

        # custom tensor names reach the graph
        _named = export_qdq(_qmodel, _sample, Path(_tmp)/'named.onnx',
                            input_names=['images'], output_names=['logits'])
        _graph = onnx.load(str(_named)).graph
        test_eq([i.name for i in _graph.input], ['images'])
        test_eq([o.name for o in _graph.output], ['logits'])

        # dynamic_batch is announced as experimental, and says so again when it could not be honoured
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            export_qdq(_qmodel, _sample, Path(_tmp)/'dynamic.onnx', dynamic_batch=True)
        _messages = [str(w.message) for w in _caught if issubclass(w.category, UserWarning)]
        assert any('experimental' in m for m in _messages), _messages
        assert any('STATIC batch' in m for m in _messages), _messages

        # an empty sample is an error, never a silent 0.0 agreement
        with ExceptionExpected(ValueError, regex="empty"):
            verify_qdq(_qmodel, _path, torch.empty(0, 3, 16, 16))

        # `n_batches` must divide `sample`: an uneven last batch is exactly what a static
        # graph rejects (torch.chunk would turn 10 inputs into 3/3/3/1)
        with ExceptionExpected(ValueError, regex="equal batches"):
            verify_qdq(_qmodel, _path, torch.randn(10, 3, 16, 16), n_batches=4)
        # ...and a shorter probe read in two batches answers the same way — on a slice whose own
        # predictions still span classes, so this reading cannot be vacuous either. Its floor is
        # looser because eight probes move the agreement in steps of 0.125: 0.875 (7 of 8) here,
        # and _MIN_SLICE_AGREEMENT leaves room for one more boundary case to round the other way.
        _slice = _spread_probe[:2 * _static_batch]
        assert len(set(_preds[:2 * _static_batch])) >= _MIN_SLICE_CLASSES, _preds[:2 * _static_batch]
        _slice_agreement = verify_qdq(_qmodel, _path, _slice, n_batches=2)
        assert _slice_agreement >= _MIN_SLICE_AGREEMENT, \
            f"the short probe disagrees too often: {_slice_agreement}"

In [ ]:
# --- end to end: the same model written with unsigned activation pairs ---
if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    import fasterai.export.onnx_exporter as _module

    def _census(path):
        "How many nodes of each operator a file carries"
        _nodes = onnx.load(str(path)).graph.node
        return {op: sum(1 for n in _nodes if n.op_type == op) for op in {n.op_type for n in _nodes}}

    with tempfile.TemporaryDirectory() as _tmp:
        _dir = Path(_tmp)
        _int8 = export_qdq(_qmodel, _sample, _dir/'int8.onnx')
        _uint8 = export_qdq(_qmodel, _sample, _dir/'uint8.onnx', activation_dtype='uint8')
        assert not (_dir/'uint8.onnx.uint8').exists(), "the rewrite left its sidecar behind"

        # the default export does not go through the rewrite at all...
        def _never(*args, **kwargs): raise AssertionError("the default export ran the uint8 rewrite")
        _write_uint8 = _module._write_uint8_activations
        try:
            _module._write_uint8_activations = _never
            assert export_qdq(_qmodel, _sample, _dir/'default.onnx').exists()
        finally:
            _module._write_uint8_activations = _write_uint8
        # ...and two default exports of one model answer the same thing, node for node
        _again = export_qdq(_qmodel, _sample, _dir/'again.onnx')
        test_eq(_output_difference(_ort_outputs(_int8, _sample), _ort_outputs(_again, _sample)), None)
        test_eq(_census(_int8), _census(_again))

        # the uint8 file is the same graph: same nodes, same per-channel weights, unsigned activations
        _stats_int8, _stats_uint8 = qdq_stats(_int8), qdq_stats(_uint8)
        test_eq(_census(_uint8), _census(_int8))
        test_eq(_stats_uint8.n_quantize, _stats_int8.n_quantize)
        test_eq(_stats_uint8.n_dequantize, _stats_int8.n_dequantize)
        test_eq(_stats_uint8.n_per_channel, _stats_int8.n_per_channel)
        test_eq(_stats_int8.n_uint8, 0)
        test_eq(_stats_int8.n_nonzero_zero_point, 0)
        assert _stats_uint8.n_uint8 > 0, _stats_uint8
        test_eq(_stats_uint8.n_nonzero_zero_point, _stats_uint8.n_uint8)

        # node by node: every activation node reads 128 as a uint8, every weight node still reads 0
        _graph = onnx.load(str(_uint8)).graph
        _constants = _graph_constants(_graph)
        for _node in _graph.node:
            if _node.op_type not in ('QuantizeLinear', 'DequantizeLinear'): continue
            _zp = numpy_helper.to_array(_constants[_node.input[2]])
            if _node.op_type == 'DequantizeLinear' and _node.input[0] in _constants:
                test_eq(_zp.dtype, np.int8)
                assert np.all(_zp == 0), _node.name
            else:
                test_eq(_zp.dtype, np.uint8)
                assert np.all(_zp == 128), _node.name

        # both files load and run, and unfused they return the same bytes — that is the contract
        for _path in (_int8, _uint8):
            test_eq(tuple(ONNXModel(_path)(_sample).shape), (_sample.shape[0], 10))
        test_eq(_output_difference(_ort_outputs(_int8, _sample, optimize=False),
                                   _ort_outputs(_uint8, _sample, optimize=False)), None)
        # fused, what is asked of it is the agreement asked of the file it was rewritten from
        assert verify_qdq(_qmodel, _uint8, _spread_probe, n_batches=_n_batches) >= _MIN_AGREEMENT

        # it composes with the rest of the export: fewer pairs to move, and an older opset
        _skipped = Quantizer(backend='pt2e', skip_activations=['features.3']).quantize(
            _TinyConvNet().eval(), _calib)
        _skip_path = export_qdq(_skipped, _sample, _dir/'skip17.onnx', opset_version=17,
                                activation_dtype='uint8')
        _skip_stats = qdq_stats(_skip_path)
        assert 0 < _skip_stats.n_quantize < _stats_int8.n_quantize, _skip_stats
        test_eq(_skip_stats.n_uint8, _skip_stats.n_nonzero_zero_point)
        assert _skip_stats.n_uint8 > 0, _skip_stats
        test_eq(next(o.version for o in onnx.load(str(_skip_path)).opset_import
                     if o.domain in ('', 'ai.onnx')), 17)

        # a refusal keeps nothing: not the rewritten file, and not the one that was produced
        def _refuse(*args, **kwargs): raise ValueError("the post-condition says no.")
        _check = _module._check_uint8_activations
        try:
            _module._check_uint8_activations = _refuse
            with ExceptionExpected(ValueError, regex="No file was kept"):
                export_qdq(_qmodel, _sample, _dir/'refused.onnx', activation_dtype='uint8')
        finally:
            _module._check_uint8_activations = _check
        assert not (_dir/'refused.onnx').exists() and not (_dir/'refused.onnx.uint8').exists()

        # ...including when what fails is the comparison, after both files are on disk
        _difference = _module._output_difference
        try:
            _module._output_difference = lambda before, after: "max |Δ| = 1.000e+00"
            with ExceptionExpected(ValueError, regex="does not compute what the produced one computes"):
                export_qdq(_qmodel, _sample, _dir/'differs.onnx', activation_dtype='uint8')
        finally:
            _module._output_difference = _difference
        assert not (_dir/'differs.onnx').exists() and not (_dir/'differs.onnx.uint8').exists()

        # a model whose activations are affine says so, in the words of the option that made it so
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _affine = Quantizer(backend='pt2e', symmetric=False).quantize(_TinyConvNet().eval(), _calib)
        with ExceptionExpected(ValueError, regex="symmetric=False"):
            export_qdq(_affine, _sample, _dir/'affine.onnx', activation_dtype='uint8')
        assert not (_dir/'affine.onnx').exists() and not (_dir/'affine.onnx.uint8').exists()

# an unknown dtype is refused before anything is written
with tempfile.TemporaryDirectory() as _tmp:
    with ExceptionExpected(ValueError, regex="Unknown activation_dtype"):
        export_qdq(nn.Linear(4, 4), torch.randn(1, 4), Path(_tmp)/'never.onnx', activation_dtype='int4')
    assert not (Path(_tmp)/'never.onnx').exists()